# 03 · Train probe on V-JEPA 2.1 features (25 games) + compare to ResNet
Light job (no heavy GPU extraction). Attaches the 25-game V-JEPA features + labels, trains the attentive probe, prints per-class accuracy and spotting mAP next to the ResNet baseline.

**Kaggle settings:**
- Accelerator: **GPU T4** (training is fast; T4 fine)
- Internet: **ON** (git clone)
- Add data: **`daniels02/vjepa-features-25games`** (the .npy features)
- Add data: **`daniels02/soccernet-25games`** (for Labels-v2.json — features dataset has no labels)

## Clone + install

In [ ]:
import os
%cd /kaggle/working
!rm -rf Football_JEPA
!git clone https://github.com/DanielSch02/Football_JEPA.git
%cd Football_JEPA
!git log --oneline -1
!pip install -q SoccerNet scipy scikit-learn matplotlib


## Merge features + labels into one working dir
Features live in `vjepa-features-25games`, labels in `soccernet-25games`. We copy both into `/kaggle/working/soccernet/england_epl/...` (canonical layout) so train/spot find them together.

In [ ]:
import glob, shutil
from pathlib import Path
WORK = "/kaggle/working/soccernet"

def stage(pattern, label):
    # Datasets may be flattened by Kaggle (no england_epl/ prefix). Key on the last
    # 3 path parts (<season>/<game>/<file>) and rebuild the canonical layout.
    n = 0
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        p = Path(src)
        season, game, fname = p.parts[-3], p.parts[-2], p.parts[-1]
        dst = Path(WORK) / "england_epl" / season / game / fname
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy(src, dst); n += 1
    print(f"staged {n} {label}")

stage("*_VJEPA21_L.npy", "feature files")
stage("Labels-v2.json", "label files")

nfeat = len(glob.glob(f"{WORK}/**/*_VJEPA21_L.npy", recursive=True))
nlab  = len(glob.glob(f"{WORK}/**/Labels-v2.json", recursive=True))
print(f"features: {nfeat}/50   labels: {nlab}/25")
assert nfeat == 50 and nlab == 25, "missing files - check both datasets are attached"


## Train the attentive probe on V-JEPA features (all 25 games)

In [ ]:
!python -m scripts.train --data_dir /kaggle/working/soccernet --feature_tag VJEPA21_L --epochs 30

## Spotting: sliding-window mAP on a valid game

In [ ]:
!python -m scripts.spot --data_dir /kaggle/working/soccernet --feature_tag VJEPA21_L     --game "england_epl/2014-2015/2015-04-11 - 19-30 Burnley 0 - 1 Arsenal" --half 1 --query Corner

## Compare to ResNet baseline
Reference (ResNet, 20 train / 5 valid): val acc ~0.69.

| Class | ResNet (20g) |
|---|---|
| Background | 0.805 |
| Foul | 0.661 |
| Kick-off | 0.690 |
| Throw-in | 0.632 |
| Clearance | 0.623 |
| Ball out of play | 0.606 |
| Shots on target | 0.605 |
| Direct free-kick | 0.652 |
| Corner | 0.364 |
| Indirect free-kick | 0.364 |
| Offside | 0.231 |

Spotting mAP (ResNet, Burnley-Arsenal H1): @5s 0.35, @10s 0.40, @30s 0.50, @60s 0.55.

Put the V-JEPA per-class table + mAP from the cells above alongside these for the head-to-head.

## (Optional) Save the trained probe to a dataset
Persists results/probe_VJEPA21_L.pt so you don't retrain each session.

In [ ]:
import json, subprocess, shutil, os
OUT = "/kaggle/working/probe_out"; os.makedirs(OUT, exist_ok=True)
shutil.copy("results/probe_VJEPA21_L.pt", f"{OUT}/probe_VJEPA21_L.pt")
json.dump({"title":"vjepa-probe-25games","id":"daniels02/vjepa-probe-25games",
           "licenses":[{"name":"CC0-1.0"}]}, open(f"{OUT}/dataset-metadata.json","w"))
r = subprocess.run(["kaggle","datasets","create","-p",OUT], capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    r = subprocess.run(["kaggle","datasets","version","-p",OUT,"-m","probe"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
